# Sprint 3 — Full MoE Pipeline (single-session, end-to-end)

Runs cache rebuild → Phase 1 → Phase 2 → Phase 3 → Eval all in one session.

## Datasets to attach
| Dataset slug | Required |
|---|---|
| `iahmedhabib/medsam-vit-b` | ✅ |
| `mabdullahi454/tb-pipeline-checkpoints` | ✅ |
| `iahmedhabib/montgomery` | ✅ |
| `iahmedhabib/shehzhenn` | ✅ |
| `usmanshams/tbx-11` | ✅ |
| `organizations/nih-chest-xrays` | optional (sanity gate) |

**GPU:** T4 × 1  
**Estimated total runtime:** ~3.5–4 hrs
- Setup: ~3 min
- Phase 0 cache build: ~45 min
- Phase 1 expert pretraining: ~50 min
- Phase 2 joint MoE: ~65 min
- Phase 3 boundary critic: ~15 min
- Evaluation: ~35 min
- Backup zip: ~3 min

## Section 1 — Setup

In [ ]:
# ── Cell 1.1: Clone repo and install dependencies ────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO_URL  = "https://github.com/AhmedHabib00/dl-project-codebase.git"  # <-- UPDATE if your fork is elsewhere
REPO_DIR  = "/kaggle/working/repo"
CACHE_DIR = "/kaggle/working/moe_cache_v2"
SAVE_DIR  = "/kaggle/working/moe_v2"
EVAL_DIR  = "/kaggle/working/eval_moe_v2"

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     os.path.join(REPO_DIR, "requirements.txt")],
    check=True
)
print("Setup complete.")

In [ ]:
# ── Cell 1.2: Verify mounts ──────────────────────────────────────────────────
from pathlib import Path

MEDSAM_CKPT = Path("/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth")
C1_ADAPTER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors")
C4_DECODER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt")

MONTGOMERY  = Path("/kaggle/input/datasets/iahmedhabib/montgomery/montgomery")
SHENZHEN    = Path("/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen")
TBX11K      = Path("/kaggle/input/datasets/usmanshams/tbx-11/TBX11K")
NIH         = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")

def check(label, path, required=True):
    status = "OK" if path.exists() else "MISSING"
    flag   = "[REQUIRED]" if required else "[optional]"
    print(f"  {flag} {label:<40}: {status}  ({path})")
    if required and not path.exists():
        raise FileNotFoundError(f"{label} not found at {path}")

check("MedSAM checkpoint",  MEDSAM_CKPT)
check("C1 LoRA adapter",    C1_ADAPTER)
check("C4 lung decoder",    C4_DECODER)
check("Montgomery dataset", MONTGOMERY)
check("Shenzhen dataset",   SHENZHEN)
check("TBX11K dataset",     TBX11K)
check("NIH-CXR14 dataset",  NIH, required=False)
print("All required mounts present.")

In [ ]:
# ── Cell 1.3: Patch configs ──────────────────────────────────────────────────
import yaml

moe_yaml_path   = Path(REPO_DIR) / "configs" / "moe.yaml"
paths_yaml_path = Path(REPO_DIR) / "configs" / "paths.yaml"

with moe_yaml_path.open() as f:
    moe_cfg = yaml.safe_load(f)

moe_cfg["component1"]["checkpoint_path"]         = str(MEDSAM_CKPT)
moe_cfg["component1"]["adapter_path"]             = str(C1_ADAPTER)
moe_cfg["component4"]["checkpoint_path"]         = str(MEDSAM_CKPT)
moe_cfg["component4"]["decoder_checkpoint_path"] = str(C4_DECODER)
moe_cfg["moe_training"]["save_dir"]              = SAVE_DIR
# Guard: ensure v2 settings are active even if YAML drifts
moe_cfg["moe_training"]["joint"]["gate_only"]            = False
moe_cfg["moe_training"]["joint"]["epochs"]               = 12
moe_cfg["moe_training"]["joint"]["lr_gate"]              = 5e-4
moe_cfg["moe_training"]["joint"]["lr_experts"]           = 5e-5
moe_cfg["moe_training"]["joint"]["load_balance_weight"]  = 0.05
moe_cfg["moe_training"]["joint"]["expert_aux_weight"]    = 0.5
moe_cfg["moe_training"]["joint"]["resume_from_moe_best"] = None
moe_cfg["moe"]["checkpoint_path"]                         = str(Path(SAVE_DIR) / "moe_best.pt")
# v2.1 supervision tightening: was 0.45 → too permissive (100% non-empty per expert).
# 0.60 forces TXV to be moderately confident before pseudo_cam path is entered.
moe_cfg.setdefault("moe_cache", {})["pseudo_threshold"]   = 0.60

with moe_yaml_path.open("w") as f:
    yaml.safe_dump(moe_cfg, f, sort_keys=False)

paths_cfg = {
    "project_root": REPO_DIR,
    "datasets": {
        "montgomery": str(MONTGOMERY),
        "shenzhen":   str(SHENZHEN),
        "tbx11k":     str(TBX11K),
        "nih_cxr14":  str(NIH) if NIH.exists() else "",
    },
}
with paths_yaml_path.open("w") as f:
    yaml.safe_dump(paths_cfg, f, sort_keys=False)

print("Configs patched.")
print(f"  Phase 2: gate_only={moe_cfg['moe_training']['joint']['gate_only']}, epochs={moe_cfg['moe_training']['joint']['epochs']}")
print(f"  Cache : pseudo_threshold={moe_cfg['moe_cache']['pseudo_threshold']}, expert confidence floor=0.35 (in source)")


## Section 2 — Phase 0: Cache Rebuild (~45 min)

Run smoke test first (~3 min). If it shows NIH ALP < 10%, proceed to full build.

In [ ]:
# ── Cell 2.1: Smoke cache build (8 images per domain, ~3 min) ───────────────
import subprocess, sys
from pathlib import Path

SMOKE_CACHE = "/kaggle/working/moe_cache_smoke"

cmd = [
    sys.executable, "-m", "scripts.cache_moe_embeddings",
    "--config",          str(Path(REPO_DIR) / "configs" / "moe.yaml"),
    "--paths",           str(Path(REPO_DIR) / "configs" / "paths.yaml"),
    "--output",          SMOKE_CACHE,
    "--limit-per-domain","8",
]
result = subprocess.run(cmd, cwd=REPO_DIR, capture_output=False)
assert result.returncode == 0, "Smoke cache build failed — check output above."
smoke_files = list(Path(SMOKE_CACHE).glob("*.pt"))
print(f"Smoke cache: {len(smoke_files)} files written to {SMOKE_CACHE}")

In [ ]:
# ── Cell 2.2: Smoke sanity check ────────────────────────────────────────────
# EXPECTED non-zero target fractions:
#   consolidation : 25–45%
#   cavity        :  8–20%  (was 0–2% with the broken ringify + whole-lung fallback)
#   fibrosis      : 15–35%
#   nodule        : 15–30%
# NIH is intentionally excluded from the cache (per CLAUDE.md). The real
# NIH "lung-painter" gate runs at eval time (Cell 6.5).

import torch
from pathlib import Path

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
stats = {e: {"total": 0, "nonzero": 0} for e in expert_names}

for p in Path(SMOKE_CACHE).glob("*.pt"):
    d = torch.load(p, weights_only=False)
    for e in expert_names:
        mask = d["expert_masks"][e]
        has_content = float(mask.sum().item()) > 0
        stats[e]["total"]   += 1
        stats[e]["nonzero"] += int(has_content)

print("Expert non-zero mask fractions in smoke cache:")
GATE_PASS = True
for e in expert_names:
    t = stats[e]["total"]
    n = stats[e]["nonzero"]
    pct = 100.0 * n / t if t else 0.0
    ok = pct > 0  # Some non-zero on at least the smoke set; full check is in Cell 2.4
    if not ok: GATE_PASS = False
    print(f"  [{'OK  ' if ok else 'WARN'}] {e:<16}: {n}/{t}  ({pct:.1f}%)")

assert GATE_PASS, "Smoke cache: every expert is empty — supervision pipeline broken."
print("\nSmoke OK. Proceeding to full build.")


In [ ]:
# ── Cell 2.3: Full cache build (1000 images per domain, ~45 min) ────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
cmd = [
    sys.executable, "-m", "scripts.cache_moe_embeddings",
    "--config",          str(Path(REPO_DIR) / "configs" / "moe.yaml"),
    "--paths",           str(Path(REPO_DIR) / "configs" / "paths.yaml"),
    "--output",          CACHE_DIR,
    "--limit-per-domain","1000",
]
result = subprocess.run(cmd, cwd=REPO_DIR)
elapsed = (time.time() - t0) / 60
assert result.returncode == 0, "Full cache build failed — check output above."

cache_files = list(Path(CACHE_DIR).glob("*.pt"))
print(f"Full cache: {len(cache_files)} files written in {elapsed:.1f} min")

In [ ]:
# ── Cell 2.4: Full cache sanity check + dataset balance ─────────────────────
# v2.1 EXPECTED RANGES with stricter thresholds (pseudo_threshold=0.6, conf=0.35):
#   consolidation : 30–60% non-empty
#   cavity        :  5–25% non-empty
#   fibrosis      : 15–40% non-empty
#   nodule        : 10–35% non-empty
# If ANY expert is > 90%, supervision is still too dense — abort.
# If ANY expert is < 2%, supervision is too sparse — also abort.

import torch
from collections import Counter
from pathlib import Path

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
dataset_counts = Counter()
expert_nonzero = {e: Counter() for e in expert_names}

for p in Path(CACHE_DIR).glob("*.pt"):
    ds = p.stem.split("__")[0]
    dataset_counts[ds] += 1
    d = torch.load(p, weights_only=False)
    for e in expert_names:
        if float(d["expert_masks"][e].sum().item()) > 0:
            expert_nonzero[e][ds] += 1

print("Dataset sample counts (balanced sampler will equalise during training):")
for ds, cnt in sorted(dataset_counts.items()):
    print(f"  {ds:<20}: {cnt}")

total = sum(dataset_counts.values())
print("\nExpert non-zero target fractions (all datasets):")
GATE_PASS = True
for e in expert_names:
    n = sum(expert_nonzero[e].values())
    pct = 100.0 * n / total if total else 0.0
    too_sparse = pct < 2.0
    too_dense  = pct > 90.0
    ok = (not too_sparse) and (not too_dense)
    if not ok: GATE_PASS = False
    flag = "OK  " if ok else ("FAIL — too sparse" if too_sparse else "FAIL — too dense (still)")
    print(f"  [{flag}] {e:<16}: {n}/{total}  ({pct:.1f}%)")

assert GATE_PASS, "Cache sanity check FAILED — supervision pipeline is wrong."
print("\nFull cache passed sanity check (within expected v2.1 ranges).")


## Section 3 — Phase 1: Expert Pretraining (~50 min)

In [ ]:
# ── Cell 3.1: Phase 1 — pretrain all 4 expert decoders ──────────────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_experts",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
        "--all",
    ],
    cwd=REPO_DIR,
)
elapsed = (time.time() - t0) / 60
print(f"Phase 1 finished in {elapsed:.1f} min  (return code: {result.returncode})")
assert result.returncode == 0, "Phase 1 failed — check output above."

# train_experts.py saves <expert>_best.pt AND <expert>_final.pt per expert (8 files total)
best_ckpts = sorted(Path(SAVE_DIR).glob("expert_*_best.pt"))
print(f"Best expert checkpoints saved ({len(best_ckpts)}):")
for p in best_ckpts:
    print(f"  {p.name}: {p.stat().st_size/1e6:.1f} MB")
assert len(best_ckpts) == 4, f"Expected 4 *_best.pt files, found {len(best_ckpts)}"


## Section 4 — Phase 2: Joint MoE Training (~65 min)

In [ ]:
# ── Cell 4.1: Phase 2 — joint training (gate + experts + fusion) ────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_moe_joint",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
    ],
    cwd=REPO_DIR,
)
elapsed = (time.time() - t0) / 60
print(f"Phase 2 finished in {elapsed:.1f} min  (return code: {result.returncode})")
assert result.returncode == 0, "Phase 2 failed — check output above."

moe_best = Path(SAVE_DIR) / "moe_best.pt"
assert moe_best.exists(), "moe_best.pt not found after Phase 2!"
print(f"moe_best.pt saved: {moe_best.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Cell 4.2: Phase 2 convergence check ─────────────────────────────────────
# Healthy: mask_loss drops by > 0.03 across 12 epochs.
# Healthy: lb_loss drops toward 1.0 (gate is specialising, not routing uniformly).
import json
from pathlib import Path

log_file = Path(SAVE_DIR) / "moe_joint_history.jsonl"
if log_file.exists():
    epochs, mask_losses, lb_losses, total_losses = [], [], [], []
    with log_file.open() as f:
        for line in f:
            e = json.loads(line)
            epochs.append(e.get("epoch"))
            mask_losses.append(e.get("mask_loss", float("nan")))
            lb_losses.append(e.get("lb_loss", float("nan")))
            total_losses.append(e.get("total", float("nan")))

    print(f"  {'Epoch':<8} {'total':<10} {'mask_loss':<12} {'lb_loss':<10}")
    for ep, tot, ml, lb in zip(epochs, total_losses, mask_losses, lb_losses):
        print(f"  {ep:<8} {tot:<10.4f} {ml:<12.4f} {lb:<10.4f}")

    if len(mask_losses) >= 2:
        delta = mask_losses[0] - mask_losses[-1]
        print(f"\nMask loss drop: {delta:.4f}  ({'OK — model learning' if delta > 0.03 else 'WARN — barely moved'})")
        lb_drop = lb_losses[0] - lb_losses[-1]
        print(f"LB loss drop:   {lb_drop:.4f}  ({'OK — gate specialising' if lb_drop > 0.05 else 'WARN — gate uniform'})")
else:
    print(f"Log file not found at {log_file} — training may still have succeeded.")


## Section 5 — Phase 3: Boundary Critic (~15 min, optional)

In [ ]:
# ── Cell 5.1: Phase 3 — boundary critic ─────────────────────────────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_boundary_critic",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
    ],
    cwd=REPO_DIR,
)
elapsed = (time.time() - t0) / 60
print(f"Phase 3 finished in {elapsed:.1f} min  (return code: {result.returncode})")
if result.returncode != 0:
    print("Phase 3 failed — boundary critic is optional; proceeding to eval.")

## Section 6 — Evaluation (~35 min)

In [ ]:
# ── Cell 6.1: Build eval-specific configs ───────────────────────────────────
import yaml
from pathlib import Path

moe_eval_cfg = yaml.safe_load(open(Path(REPO_DIR) / "configs" / "moe.yaml"))
moe_eval_cfg["moe"]["checkpoint_path"] = str(Path(SAVE_DIR) / "moe_best.pt")

moe_eval_file = Path(EVAL_DIR) / "moe_eval.yaml"
with moe_eval_file.open("w") as f:
    yaml.safe_dump(moe_eval_cfg, f, sort_keys=False)

paths_eval_cfg = {
    "project_root": REPO_DIR,
    "datasets": {
        "montgomery": str(MONTGOMERY),
        "shenzhen":   str(SHENZHEN),
        "tbx11k":     str(TBX11K),
        "nih_cxr14":  str(NIH) if NIH.exists() else "",
    },
}
paths_eval_file = Path(EVAL_DIR) / "paths_eval.yaml"
with paths_eval_file.open("w") as f:
    yaml.safe_dump(paths_eval_cfg, f, sort_keys=False)

print("Eval configs written.")

In [ ]:
# ── Cell 6.2: Run MoE evaluation ────────────────────────────────────────────
# tbx_list_name="all_trainval.txt" is the safe default that exists in the dataset.
# (test.txt may not be present in this Kaggle mount; falls back to dir walk.)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.evaluation.moe_eval import run_moe_evaluation

summary = run_moe_evaluation(
    moe_config_path  = Path(EVAL_DIR) / "moe_eval.yaml",
    paths_config_path= Path(EVAL_DIR) / "paths_eval.yaml",
    output_dir       = Path(EVAL_DIR),
    limit_per_domain = 200,
    tbx_list_name    = "all_trainval.txt",
    repo_root        = Path(REPO_DIR),
)
print("\nMoE evaluation complete.")


In [ ]:
# ── Cell 6.3: Component metrics ─────────────────────────────────────────────
import pandas as pd
from pathlib import Path

comp_df = pd.read_csv(Path(EVAL_DIR) / "moe_components.csv")
comp_df["value"] = comp_df["value"].apply(
    lambda x: f"{float(x):.4f}" if pd.notna(x) and str(x) != "nan" else "N/A"
)
print("=== MoE COMPONENT-LEVEL METRICS ===")
print(comp_df[["metric","dataset","value","n","notes"]].to_string(index=False))

In [ ]:
# ── Cell 6.4: System metrics ────────────────────────────────────────────────
import pandas as pd
from pathlib import Path

sys_df = pd.read_csv(Path(EVAL_DIR) / "moe_system.csv")
sys_df["value"] = sys_df["value"].apply(
    lambda x: f"{float(x):.4f}" if pd.notna(x) and str(x) != "nan" else "N/A"
)
print("=== MoE SYSTEM-LEVEL METRICS ===")
print(sys_df[["metric","dataset","value","n","notes"]].to_string(index=False))

In [ ]:
# ── Cell 6.5: NIH ALP sanity gate ───────────────────────────────────────────
# If NIH ALP mean > 8%, experts are still painting lungs. The fix didn't take.
import pandas as pd
from pathlib import Path

pi_df  = pd.read_csv(Path(EVAL_DIR) / "moe_per_image.csv")
nih_df = pi_df[pi_df["dataset_id"] == "nih_cxr14"]

if nih_df.empty:
    print("NIH-CXR14 not in eval (dataset not attached). Sanity gate skipped.")
else:
    alp_mean = nih_df["alp"].mean()
    alp_p95  = nih_df["alp"].quantile(0.95)
    ok = alp_mean < 8.0
    print(f"NIH-CXR14 ALP (n={len(nih_df)}):")
    print(f"  mean : {alp_mean:.2f}%  ({'OK' if ok else 'FAIL — experts painting lungs'})")
    print(f"  p95  : {alp_p95:.2f}%")

In [ ]:
# ── Cell 6.6: Bootstrap 95% CI on Timika AUROC ──────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score

pi_df = pd.read_csv(Path(EVAL_DIR) / "moe_per_image.csv")

def bootstrap_auroc(y_true, y_score, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    auc_vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, ys = y_true[idx], y_score[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            auc_vals.append(roc_auc_score(yt, ys))
        except Exception:
            pass
    if not auc_vals:
        return float("nan"), float("nan"), float("nan")
    return float(np.mean(auc_vals)), float(np.percentile(auc_vals, 2.5)), float(np.percentile(auc_vals, 97.5))

TARGET = {"montgomery": 0.60, "shenzhen": 0.70, "tbx11k": 0.70}

print("Timika AUROC with 95% bootstrap CI (2000 resamples):")
print(f"  {'Dataset':<14} {'AUROC':>8}  {'95% CI':>16}  {'n_total':>8}  {'n_TB+':>7}  {'Status'}")
print("  " + "-" * 75)
for dom in ["montgomery", "shenzhen", "tbx11k"]:
    sub = pi_df[(pi_df["dataset_id"] == dom) & pi_df["tb_label"].notna()].copy()
    if sub.empty or sub["tb_label"].nunique() < 2:
        print(f"  {dom:<14}   N/A      (no/single-class labels, n={len(sub)})")
        continue
    y_true  = sub["tb_label"].astype(int).values
    y_score = sub["timika_score"].values
    mean_auc, lo, hi = bootstrap_auroc(y_true, y_score)
    n_pos = int(y_true.sum())
    target = TARGET.get(dom, 0.70)
    met = "PASS" if mean_auc >= target else "FAIL"
    print(f"  {dom:<14}  {mean_auc:>6.4f}   [{lo:.4f}–{hi:.4f}]  {len(sub):>8}  {n_pos:>7}  [{met} (≥{target})]")

In [ ]:
# ── Cell 6.7: Sprint 3 target table ─────────────────────────────────────────
import pandas as pd
from pathlib import Path

sys_df = pd.read_csv(Path(EVAL_DIR) / "moe_system.csv")
sys_df["value"] = pd.to_numeric(sys_df["value"], errors="coerce")

TARGETS = {
    ("timika_auroc",  "montgomery"): ("≥ 0.60", 0.60),
    ("timika_auroc",  "shenzhen"):   ("≥ 0.70", 0.70),
    ("tb_head_auroc", "shenzhen"):   ("≥ 0.85", 0.85),
    ("tb_head_auroc", "montgomery"): ("≥ 0.80", 0.80),
}

print(f"{'Metric':<25} {'Dataset':<14} {'Achieved':>10}  {'Target':>8}  {'Status'}")
print("-" * 70)
for (metric, dataset), (target_str, target_val) in TARGETS.items():
    row = sys_df[(sys_df["metric"] == metric) & (sys_df["dataset"] == dataset)]
    if row.empty or pd.isna(row["value"].values[0]):
        print(f"{metric:<25} {dataset:<14}      N/A      {target_str:>8}  N/A")
    else:
        val = float(row["value"].values[0])
        status = "PASS" if val >= target_val else f"FAIL (gap={target_val-val:.4f})"
        print(f"{metric:<25} {dataset:<14}  {val:>8.4f}    {target_str:>8}  {status}")

## Section 7 — Backup Artifacts (zip cache + MoE weights)

In [ ]:
# ── Cell 7.1: Zip MoE weights and eval results for download ────────────────
# NOTE: The cache itself (CACHE_DIR) is many GB. Set ZIP_CACHE=True only if
# you actually need to download it. The MoE weights and eval results are
# small enough to zip and download safely.
import shutil
from pathlib import Path
from IPython.display import FileLink, display

ZIP_CACHE = False  # Set True only if you really need to download the cache

print("Zipping MoE weights ...")
moe_zip = shutil.make_archive("/kaggle/working/moe_v2", "zip", SAVE_DIR)
print(f"  {moe_zip}  ({Path(moe_zip).stat().st_size/1e6:.1f} MB)")

print("Zipping eval results ...")
eval_zip = shutil.make_archive("/kaggle/working/eval_moe_v2", "zip", EVAL_DIR)
print(f"  {eval_zip}  ({Path(eval_zip).stat().st_size/1e6:.1f} MB)")

print("\nDownload links:")
display(FileLink(moe_zip))
display(FileLink(eval_zip))

if ZIP_CACHE:
    print("\nZipping cache (large, may take 5+ min) ...")
    cache_zip = shutil.make_archive("/kaggle/working/moe_cache_v2", "zip", CACHE_DIR)
    print(f"  {cache_zip}  ({Path(cache_zip).stat().st_size/1e6:.1f} MB)")
    display(FileLink(cache_zip))

print("\nAll done. Outputs in /kaggle/working/.")
print("To save as Kaggle dataset: click the 3-dot menu next to each .zip in the Output panel.")
